In [1]:
# ============================================================================
# CELDA 1: IMPORTACIONES Y CONFIGURACIÓN DE RUTAS
# ============================================================================

import sys
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ajustar la ruta para que Python encuentre los módulos del proyecto
NOTEBOOK_DIR = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# Importar módulos del proyecto
from core.parameters import SystemParameters, OptimizationParameters
from core.data_loader import SystemData
from core.system_cache import SystemCache
from core.power_flow import power_flow_hourly_optimized  # ← IMPORTANTE
from core.lsi import calculate_lsi_base_case
from algorithms.pso import pso_algorithm
from algorithms.woa import woa_algorithm
from algorithms.woa_pso import woa_pso_hybrid
from algorithms.pso_woa import pso_woa_hybrid
from algorithms.ga import ga_algorithm
from utils.io_excel import save_results_to_excel
from utils.summary import save_global_summary, generate_summary

print("Módulos importados correctamente")
print(f"Directorio del proyecto: {PROJECT_ROOT}")

Módulos importados correctamente
Directorio del proyecto: G:\Mi unidad\TESIS\CODIGO\codigo\proyecto


In [2]:
# ============================================================================
# CELDA 2: CONFIGURACIÓN DE LA EJECUCIÓN
# ============================================================================

SISTEMA = 'IEEE33'  # 'IEEE33' o 'IEEE69'
ALGORITMO = 'PSOWOA'  # 'PSO', 'WOA', 'WOAPSO', 'PSOWOA', 'GA'
NC_MAX = 4  # Número máximo de DGs y BCs
N_AGENTS = 30  # Tamaño de la población
MAX_ITER = 50  # Número de iteraciones
NUM_EXECUTIONS = 2  # Número de ejecuciones independientes
USE_LSI = True  # Usar análisis LSI para sembrar candidatos

OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'optimizacion', ALGORITMO, SISTEMA)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Sistema: {SISTEMA}")
print(f"Algoritmo: {ALGORITMO}")
print(f"Número de dispositivos (NC_max): {NC_MAX}")
print(f"Agentes: {N_AGENTS} | Iteraciones: {MAX_ITER}")
print(f"Ejecuciones: {NUM_EXECUTIONS}")
print(f"Usar LSI: {USE_LSI}")
print(f"Directorio de salida: {OUTPUT_DIR}")

Sistema: IEEE33
Algoritmo: PSOWOA
Número de dispositivos (NC_max): 4
Agentes: 30 | Iteraciones: 50
Ejecuciones: 2
Usar LSI: True
Directorio de salida: G:\Mi unidad\TESIS\CODIGO\codigo\proyecto\outputs\optimizacion\PSOWOA\IEEE33


In [3]:
# ============================================================================
# CELDA 3: INICIALIZACIÓN DEL SISTEMA Y CÁLCULO DEL CASO BASE
# ============================================================================

# Cargar parámetros de optimización desde JSON
OPT_PARAMS_PATH = os.path.join(PROJECT_ROOT, 'data', 'parameters', 'optimization_parameters.json')
opt_params = OptimizationParameters.from_json(OPT_PARAMS_PATH)

# Parámetros del sistema eléctrico
sys_params = SystemParameters()

# Cargar datos de la red
system_data = SystemData(system_type=SISTEMA)

# Crear caché con estructuras precalculadas
cache = SystemCache(sys_params, system_data)

# Calcular pérdidas del caso base (sin compensación)
print("Calculando caso base (sin compensación)...")
PTloss_ref_hourly = np.zeros(24)
for hora in range(24):
    res = power_flow_hourly_optimized(None, hora, np.zeros(NC_MAX), 0, cache,
                                      with_compensation=False)
    PTloss_ref_hourly[hora] = res['PTloss']

print(f"Pérdida promedio caso base: {np.mean(PTloss_ref_hourly):.4f} kW")

# Calcular LSI si se requiere
top_pv_buses = None
top_bc_buses = None
if USE_LSI:
    print("\nCalculando índices LSI para candidatos...")
    top_pv_buses, top_bc_buses = calculate_lsi_base_case(cache)


Calculando caso base (sin compensación)...
Pérdida promedio caso base: 72.4299 kW

Calculando índices LSI para candidatos...

Calculando LSI para caso base...
  PV candidatos (LSI1): [24 25 30  8  7 14  4  3  2 18 32 23 20 22 21 19]
  BC candidatos (LSI2): [30 24 25  8  7  4 14  3  2 23 32 18 13  5  6 20]


In [4]:
# ============================================================================
# CELDA 4: SELECCIÓN DEL ALGORITMO Y EJECUCIÓN
# ============================================================================

algo_map = {
    'PSO': pso_algorithm,
    'WOA': woa_algorithm,
    'WOAPSO': woa_pso_hybrid,
    'PSOWOA': pso_woa_hybrid,
    'GA': ga_algorithm,
}

if ALGORITMO not in algo_map:
    raise ValueError(f"Algoritmo '{ALGORITMO}' no reconocido. Opciones: {list(algo_map.keys())}")

all_executions = []

for ejec in range(1, NUM_EXECUTIONS + 1):
    print("\n" + "=" * 70)
    print(f"EJECUCIÓN {ejec}/{NUM_EXECUTIONS} - {ALGORITMO} - {SISTEMA}")
    print("=" * 70)

    result = algo_map[ALGORITMO](
        NC_max=NC_MAX,
        opt_params=opt_params,
        PTloss_ref_hourly=PTloss_ref_hourly,
        cache=cache,
        n_agents=N_AGENTS,
        max_iter=MAX_ITER,
        top_pv_buses=top_pv_buses,
        top_bc_buses=top_bc_buses
    )

    excel_filename = os.path.join(OUTPUT_DIR, f'Ejecucion_{ejec}_{ALGORITMO}_{SISTEMA}.xlsx')
    save_results_to_excel(
        opt_result=result,
        best_x=result['best_position'],
        NC_max=NC_MAX,
        opt_params=opt_params,
        PTloss_ref_hourly=PTloss_ref_hourly,
        cache=cache,
        base_case_results={},
        filename=excel_filename
    )

    resumen = generate_summary(
        best_x=result['best_position'],
        NC_max=NC_MAX,
        opt_params=opt_params,
        PTloss_ref_hourly=PTloss_ref_hourly,
        cache=cache,
        base_case_results={},
        opt_result=result
    )
    all_executions.append(resumen)

    print(f"\nResumen ejecución {ejec}:")
    print(f"  Función Objetivo: {resumen['Funcion_Objetivo']:.6f} M$")
    print(f"  PV: {resumen['PV_Instalados']} | BC: {resumen['BC_Instalados']}")
    print(f"  Reducción pérdidas: {resumen['Reduccion_Perdidas_Porcentaje']:.2f}%")


EJECUCIÓN 1/2 - PSOWOA - IEEE33
OPTIMIZACIÓN HÍBRIDA PSO-WOA - IEEE33
Agentes: 30 | Iteraciones: 50 | Switch PSO→WOA: 40
Iter   1 [PSO] | Best: 240762.834065 M$
Iter  10 [PSO] | Best: 1.823147 M$
Iter  20 [PSO] | Best: 1.434768 M$
Iter  30 [PSO] | Best: 1.367959 M$
Iter  40 [PSO] | Best: 1.342793 M$
Iter  50 [WOA] | Best: 1.342319 M$
COMPLETADO EN 34.91s | MEJOR: 1.342319 M$

Archivo guardado: G:\Mi unidad\TESIS\CODIGO\codigo\proyecto\outputs\optimizacion\PSOWOA\IEEE33\Ejecucion_1_PSOWOA_IEEE33.xlsx

Resumen ejecución 1:
  Función Objetivo: 1.342319 M$
  PV: 4 | BC: 3
  Reducción pérdidas: 40.77%

EJECUCIÓN 2/2 - PSOWOA - IEEE33
OPTIMIZACIÓN HÍBRIDA PSO-WOA - IEEE33
Agentes: 30 | Iteraciones: 50 | Switch PSO→WOA: 40
Iter   1 [PSO] | Best: 191810.780382 M$
Iter  10 [PSO] | Best: 1.468713 M$
Iter  20 [PSO] | Best: 1.360598 M$
Iter  30 [PSO] | Best: 1.360598 M$
Iter  40 [PSO] | Best: 1.329505 M$
Iter  50 [WOA] | Best: 1.328184 M$
COMPLETADO EN 41.34s | MEJOR: 1.328184 M$

Archivo guardad

In [5]:
# ============================================================================
# CELDA 5: GUARDAR RESUMEN GLOBAL
# ============================================================================

if NUM_EXECUTIONS > 1:
    save_global_summary(all_executions, ALGORITMO, SISTEMA, OUTPUT_DIR)
    print(f"\nResumen global guardado en {OUTPUT_DIR}")



Resumen global guardado: G:\Mi unidad\TESIS\CODIGO\codigo\proyecto\outputs\optimizacion\PSOWOA\IEEE33\Resumen_PSOWOA_IEEE33.xlsx

Resumen global guardado en G:\Mi unidad\TESIS\CODIGO\codigo\proyecto\outputs\optimizacion\PSOWOA\IEEE33
